In [ ]:
import pandas as pd
excel_file = '/content/한국어 학습용 어휘 목록.xls'
df = pd.read_excel(excel_file)
csv_file = '/content/Data.csv'
df.to_csv(csv_file, index=False)


In [ ]:
import pandas as pd

# Load the CSV file
csv_file = '/content/Data.csv'
df = pd.read_csv(csv_file)

print(df)


           순위     단어 품사   풀이 등급
0      1195.0     가게  명  NaN  A
1       898.0   가격03  명   價格  B
2      2986.0   가구03  명   家口  C
3      7434.0   가구04  명   家具  B
4      4964.0  가까워지다  동  NaN  B
...       ...    ... ..  ... ..
5960    489.0    힘들다  형  NaN  A
5961  10357.0  힘들어하다  동  NaN  C
5962   3305.0    힘쓰다  동  NaN  C
5963   9013.0    힘없이  부  NaN  C
5964   3846.0    힘차다  형  NaN  C

[5965 rows x 5 columns]


In [ ]:
df.columns

Index(['순위', '단어', '품사', '풀이', '등급'], dtype='object')

In [ ]:
a = df["등급"]

In [ ]:
import random

def map_grade(grade):
    if grade == 'A':
        return random.randint(1, 2)
    elif grade == 'B':
        return random.randint(3, 4)
    elif grade == 'C':
        return random.randint(5, 6)
    else:
        return None  # in case of unexpected value

# Apply the function to the '등급' column
df['등급'] = df['등급'].apply(map_grade)

print(df)

           순위     단어 품사   풀이  등급
0      1195.0     가게  명  NaN   2
1       898.0   가격03  명   價格   3
2      2986.0   가구03  명   家口   6
3      7434.0   가구04  명   家具   4
4      4964.0  가까워지다  동  NaN   4
...       ...    ... ..  ...  ..
5960    489.0    힘들다  형  NaN   2
5961  10357.0  힘들어하다  동  NaN   5
5962   3305.0    힘쓰다  동  NaN   5
5963   9013.0    힘없이  부  NaN   5
5964   3846.0    힘차다  형  NaN   5

[5965 rows x 5 columns]


In [ ]:
a = df

# Model


 [KR-Medium](https://www.google.com/url?q=https%3A%2F%2Fhuggingface.co%2Fsnunlp%2FKR-Medium)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

df = a
tokenizer = AutoTokenizer.from_pretrained("snunlp/KR-Medium", do_lower_case=False)


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

def tokenize_data(text):
    return tokenizer(text, padding='max_length', truncation=True, max_length=128)

df['input_ids'] = df['단어'].apply(lambda x: tokenize_data(x)['input_ids'])
df['attention_mask'] = df['단어'].apply(lambda x: tokenize_data(x)['attention_mask'])

label_encoder = LabelEncoder()
df['encoded_labels'] = label_encoder.fit_transform(df['등급'])


class MyDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
        self.input_ids = list(dataframe['input_ids'])
        self.attention_mask = list(dataframe['attention_mask'])
        self.labels = list(dataframe['encoded_labels'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = MyDataset(train_df)
eval_dataset = MyDataset(eval_df)


model = AutoModelForSequenceClassification.from_pretrained("snunlp/KR-Medium", num_labels=len(df['encoded_labels'].unique()))


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=7,
    per_device_train_batch_size=8,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
results = trainer.evaluate()
print(f"Training Accuracy: {results['eval_accuracy']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

steps = [log["step"] for log in logs if "loss" in log]
losses = [log["loss"] for log in logs if "loss" in log]

plt.plot(steps, losses,'o-', linewidth=1)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training Loss over Time")
plt.grid(True)
plt.show()

In [ ]:
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/vocab.txt',
 './fine_tuned_model/added_tokens.json',
 './fine_tuned_model/tokenizer.json')

In [ ]:
import shutil
import zipfile
import os

def zip_folder(folder_path, output_zip_path):
    """Zips the contents of a folder into a zip file.

    Args

        folder_path: The path to the folder to be zipped.
        output_zip_path: The path to the output zip file.
    """

    with zipfile.ZipFile(output_zip_path, "w") as zip_file:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                zip_file.write(file_path)

if __name__ == "__main__":
    folder_path = "/content/fine_tuned_model" #Souce folder
    output_zip_path = "/content/komodel.zip" #zip file

    zip_folder(folder_path, output_zip_path)


In [ ]:
!unzip "/content/komodel.zip" -d "/content/komodel/"

Archive:  /content/komodel.zip
replace /content/komodel/content/fine_tuned_model/model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
 extracting: /content/komodel/content/fine_tuned_model/model.safetensors  
replace /content/komodel/content/fine_tuned_model/vocab.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("/content/fine_tuned_model")
model = AutoModelForSequenceClassification.from_pretrained("/content/fine_tuned_model")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/fine_tuned_model'. Use `repo_type` argument if needed.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

label_encoder = LabelEncoder()
label_encoder.fit(df['등급'])


LabelEncoder()

In [ ]:
def predict_grade(word):
    inputs = tokenizer(word, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=-1).item()

    label = label_encoder.inverse_transform([predicted_class])[0]
    return label


In [ ]:
test_words = input("Word:") # <---

result = predict_grade(test_words)
print(f"'{test_words}' → : {result}")


Word:가
'가' → : 3


In [ ]:
from sklearn.metrics import accuracy_score

# y_true: list or array of true labels
# y_pred: list or array of predicted labels

# Example:
y_true = [0, 1, 1, 0, 1]  # actual labels
y_pred = [0, 0, 1, 0, 1]  # predicted by your model

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8000
